# ChromaDB Basics

**ChromaDB** is an open-source vector database built for storing and searching
*embeddings* — numeric representations of text. It lets you ask questions in
natural language and find the most semantically similar documents.

In this notebook you will learn how to:

1. Create a persistent Chroma client and a collection.
2. Add documents (Chroma embeds them automatically).
3. Run similarity searches.
4. Filter results using metadata.
5. Update, retrieve, and delete documents.
6. Manage collections (rename, list, delete) and reset the database.

> **What is an embedding?** An embedding maps text to a vector of numbers such
> that texts with similar *meaning* end up close together in vector space.
> Searching then becomes a nearest-neighbour problem.

## 1. Imports

We need the `chromadb` client, `os`/`tempfile` to build a throwaway storage
path, and `Settings` to allow resetting the database during the demo.

In [1]:
import chromadb
from chromadb.config import Settings
import os
import tempfile

## 2. Create a client

Chroma has two common client types:

- **`PersistentClient`** – stores data on disk (used here).
- **`HttpClient`** – talks to a Chroma server over HTTP (commented out below).

We point the persistent client at a folder inside the system temp directory so
the demo is self-contained and can be safely wiped.

In [2]:
collection_name = "Students"

client = chromadb.PersistentClient(
    path=os.path.join(tempfile.gettempdir(), "chroma_db"),
    settings=Settings(allow_reset=True),
)

# Alternatively, connect to a running Chroma server:
# client = chromadb.HttpClient(
#     host="localhost",
#     port=8000,
#     settings=Settings(allow_reset=True),
# )

## 3. Create or get a collection

A **collection** is the container that holds documents, their metadata, and
their embeddings — roughly analogous to a table in a relational database.

`get_or_create_collection` is idempotent: it creates the collection the first
time and simply returns it on later runs.

See the docs: https://docs.trychroma.com/docs/collections/configure#python

In [3]:
collection = client.get_or_create_collection(name=collection_name)

## 4. Prepare some documents

We will work with three short, unrelated pieces of text: a student bio, a club
description, and a university description. Having distinct topics makes it easy
to see how semantic search and metadata filtering behave.

In [4]:
student_info = """
Alexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,
is a member of the programming and chess clubs who enjoys pizza, swimming, and hiking
in her free time in hopes of working at a tech company after graduating from the University of Washington.
"""

club_info = """
The university chess club provides an outlet for students to come together and enjoy playing
the classic strategy game of chess. Members of all skill levels are welcome, from beginners learning
the rules to experienced tournament players. The club typically meets a few times per week to play casual games,
participate in tournaments, analyze famous chess matches, and improve members' skills.
"""

university_info = """
The University of Washington, founded in 1861 in Seattle, is a public research university
with over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.
As the flagship institution of the six public universities in Washington state,
UW encompasses over 500 buildings and 20 million square feet of space,
including one of the largest library systems in the world.
"""

## 5. Add documents to the collection

`add()` requires three parallel lists:

- `documents` – the raw text to be embedded and stored.
- `metadatas` – key/value pairs used for filtering later.
- `ids` – a unique string ID for every document.

> **Default embedding function:** Chroma uses the Sentence Transformers
> `all-MiniLM-L6-v2` model to embed text. It runs **locally** on your machine
> and downloads the model files automatically the first time.
>
> Docs: https://docs.trychroma.com/docs/embeddings/embedding-functions

In [5]:
collection.add(
    documents=[student_info, club_info, university_info],
    metadatas=[
        {"source": "student info"},
        {"source": "club info"},
        {"source": "university info"},
    ],
    ids=["id1", "id2", "id3"],
)

## 6. Similarity search

`query()` converts your natural-language question into an embedding and returns
the documents whose embeddings are closest to it. Here we ask for just the top
result.

Notice the returned payload contains `ids`, `documents`, `metadatas`, and
`distances` (a smaller distance means a closer / more similar match).

In [6]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=1,
)
print("Query 1: ", results)

Query 1:  {'ids': [['id1']], 'embeddings': None, 'documents': [['\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'student info'}]], 'distances': [[1.2946666479110718]]}


## 7. Filtering with metadata

Similarity search alone can return results from any topic. Use the `where`
parameter to constrain the search to documents matching metadata conditions.

Below we ask for 2 results but restrict them to documents whose
`source` is `"student info"`.

In [7]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    where={"source": "student info"},  # only return documents with this metadata
)
print("Query 2: ", results)

Query 2:  {'ids': [['id1']], 'embeddings': None, 'documents': [['\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'student info'}]], 'distances': [[1.2946666479110718]]}


### Combining filters with `$or` / `$and`

Chroma supports logical operators for more expressive filters. Here we search
for `"university"` but accept documents from **either** the student or the
university source.

In [8]:
results = collection.query(
    query_texts=["university"],
    n_results=5,
    where={
        "$or": [
            {"source": "student info"},
            {"source": "university info"},
        ]
    },
)
print("Query 3: ", results)

Query 3:  {'ids': [['id3', 'id1']], 'embeddings': None, 'documents': [['\nThe University of Washington, founded in 1861 in Seattle, is a public research university\nwith over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.\nAs the flagship institution of the six public universities in Washington state,\nUW encompasses over 500 buildings and 20 million square feet of space,\nincluding one of the largest library systems in the world.\n', '\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'university info'}, {'source': 'student info'}]], 'distances': [[1.1075575351715088, 1.385632872581482]]}


## 8. Including embeddings and distances

By default `query()` does not return the raw vectors. Pass `include` to request
`embeddings`, `documents`, and/or `distances` explicitly — useful when you want
to inspect or reuse the vectors yourself.

In [9]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    include=["embeddings", "documents", "distances"],
)
print("Query 4: ", results)

Query 4:  {'ids': [['id1', 'id2']], 'embeddings': [array([[-3.42186019e-02,  1.53967049e-02, -1.70410573e-02,
         4.85090129e-02, -4.30887341e-02, -3.49062262e-04,
         1.11857228e-01, -1.35589140e-02, -4.07729624e-03,
        -1.29493577e-02, -1.08448923e-01, -2.81905159e-02,
        -6.44052327e-02,  2.02892534e-02, -1.83084402e-02,
         7.67185241e-02,  2.00441983e-02,  7.45956833e-03,
        -6.27691969e-02, -1.21103801e-01, -1.10205069e-01,
        -9.30497125e-02,  1.71378590e-02,  3.06136557e-03,
        -2.04835902e-04,  3.19657438e-02,  6.24834001e-02,
        -2.02021748e-02, -4.48452942e-02,  6.20209761e-02,
        -5.34061678e-02, -2.63791066e-03,  3.05590518e-02,
         6.97120577e-02,  4.50212322e-02,  8.20889920e-02,
         6.09008558e-02, -3.03317066e-02,  8.87826830e-03,
         3.19182053e-02, -3.22156139e-02,  1.71087366e-02,
         3.98923643e-03,  9.67202485e-02, -3.07201259e-02,
        -5.45867831e-02, -6.89935312e-02, -7.65794069e-02,
     

## 9. Updating a document

`update()` replaces the document text for a given ID. Chroma re-embeds the new
text automatically, so subsequent searches reflect the change.

Here we swap the student's full bio for a shorter version and re-run the query.

In [10]:
collection.update(
    ids=["id1"],
    documents=["Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA"],
    metadatas=[{"source": "student info"}],
)
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
)
print("Query 5: ", results)

Query 5:  {'ids': [['id1', 'id2']], 'embeddings': None, 'documents': [['Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA', "\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous chess matches, and improve members' skills.\n"]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'student info'}, {'source': 'club info'}]], 'distances': [[1.0592992305755615, 1.395403265953064]]}


## 10. Updating metadata

A few important gotchas:

- Chroma **replaces the entire metadata object** for an ID — it does **not**
  merge keys. Always pass every field you want to keep.
- Metadata values must be **strings, integers, floats, or booleans**.
  Nested dictionaries or lists are not supported.

In [11]:
collection.update(
    ids=["id1"],  # the ID of the document you want to update
    metadatas=[
        {"source": "student info", "version": 2.0, "tags": ["python", "ai", "database"]}
    ],  # new metadata (replaces the old object)
)

## 11. Retrieving by ID and filtering

`get()` retrieves documents **without** doing a similarity search. Use it when
you already know what you want — by ID, or via metadata filters.

- `get(ids=[...])` fetches specific documents.
- `get(where=...)` filters by metadata.
- The `$contains` operator checks membership inside a list-valued metadata field.

In [12]:
# Get by ID
results = collection.get(ids=["id1"])
print("Get by ID: ", results)

# Filter by metadata
results = collection.get(where={"source": "student info"})
print("Filter by metadata 1: ", results)

# Filter by tag (membership test inside a list)
results = collection.get(where={"tags": {"$contains": "python"}})
print("Filter by metadata 2: ", results)

Get by ID:  {'ids': ['id1'], 'embeddings': None, 'documents': ['Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'tags': ['python', 'ai', 'database'], 'version': 2.0, 'source': 'student info'}]}
Filter by metadata 1:  {'ids': ['id1'], 'embeddings': None, 'documents': ['Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'student info', 'tags': ['python', 'ai', 'database'], 'version': 2.0}]}
Filter by metadata 2:  {'ids': ['id1'], 'embeddings': None, 'documents': ['Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'tags': ['python', 'ai', 'database'], 'version': 2.0, 'source': 'student info'}]}


## 12. Inspecting embeddings

Embeddings are just lists of floats. Let's look at the vectors returned by a
query, and then fetch the vectors for **all** documents in the collection.

In [13]:
query_results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    include=["embeddings", "documents", "distances"],  # request embeddings in query output
)
print("View embeddings: ", query_results)

View embeddings:  {'ids': [['id1', 'id2']], 'embeddings': [array([[-4.50674593e-02, -1.39631741e-02, -4.18995954e-02,
        -3.27188894e-02, -7.93402717e-02,  1.61406323e-02,
         1.61813926e-02,  4.61544059e-02, -2.18480220e-03,
         3.86094227e-02, -3.22900899e-02, -4.22492959e-02,
        -7.78574720e-02,  1.11621814e-02, -6.82541579e-02,
         7.17114983e-03, -6.83525205e-02,  3.18681262e-02,
        -8.99454206e-03, -4.42172736e-02, -8.24618712e-02,
         2.90755127e-02, -1.47390980e-02,  3.39503810e-02,
         7.84361735e-02, -7.96573795e-03,  2.09888630e-02,
        -4.71384712e-02, -5.50323725e-02,  4.64934893e-02,
        -1.78024359e-02,  2.17338260e-02,  1.45010119e-02,
         7.59510770e-02,  2.41132360e-02,  2.03049779e-02,
         1.30152786e-02,  3.00397407e-02, -3.00088543e-02,
         1.49036655e-02, -6.42017424e-02, -5.35866208e-02,
        -1.01466859e-02,  4.97970432e-02,  1.17468983e-02,
        -1.17518194e-01, -1.76695455e-02, -4.99595180e-0

In [14]:
# Fetch all documents and their vector embeddings
all_data = collection.get(include=["embeddings", "documents"])

all_vectors = all_data["embeddings"]
print("All vectors: ", all_vectors)

All vectors:  [[-0.04506746 -0.01396317 -0.0418996  ... -0.08546233 -0.04060214
   0.01908717]
 [ 0.02532914 -0.02509276 -0.00254027 ...  0.03525016 -0.11900327
  -0.06710915]
 [ 0.08754651  0.01729166 -0.04267624 ... -0.00880425  0.01187562
   0.02228451]]


## 13. Deleting a document

`delete(ids=[...])` removes documents from the collection. After deleting `id1`,
re-running the same query shows that the student document is gone.

In [15]:
collection.delete(ids=["id1"])

results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
)
print("Query after delete: ", results)

Query after delete:  {'ids': [['id2', 'id3']], 'embeddings': None, 'documents': [["\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous chess matches, and improve members' skills.\n", '\nThe University of Washington, founded in 1861 in Seattle, is a public research university\nwith over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.\nAs the flagship institution of the six public universities in Washington state,\nUW encompasses over 500 buildings and 20 million square feet of space,\nincluding one of the largest library systems in the world.\n']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'club info'},

## 14. Counting and listing everything

`count()` returns the number of documents, and `get()` with no arguments
returns them all.

In [16]:
print("Count of docs: ", collection.count())
print("All docs: ", collection.get())

Count of docs:  2
All docs:  {'ids': ['id2', 'id3'], 'embeddings': None, 'documents': ["\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous chess matches, and improve members' skills.\n", '\nThe University of Washington, founded in 1861 in Seattle, is a public research university\nwith over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.\nAs the flagship institution of the six public universities in Washington state,\nUW encompasses over 500 buildings and 20 million square feet of space,\nincluding one of the largest library systems in the world.\n'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'club info'}, {'source'

## 15. Renaming a collection

`modify(name=...)` renames a collection in place. We first make sure no stale
collection with the target name exists, then rename and list the collections.

In [17]:
if "chroma_info" in [r.name for r in client.list_collections()]:
    client.delete_collection("chroma_info")

collection.modify(name="chroma_info")

# List all collections
print("List collections: ", client.list_collections())

List collections:  [Collection(name=chroma_info)]


## 16. Deleting a collection

Deleting a collection removes it and all of its documents/embeddings.

In [18]:
client.delete_collection(name="chroma_info")
print("List collections (after deletion): ", client.list_collections())

List collections (after deletion):  []


## 17. Resetting the database

`reset()` wipes **all** collections and data. It is handy for demos and tests
but only works with clients that allow it (not a remote `HttpClient`).

> In production, think twice before calling `reset()`!

In [19]:
client.reset()
print("Collections after reset: ", client.list_collections())

Collections after reset:  []


## Summary

You have now seen the core ChromaDB workflow:

| Operation | Method |
| --- | --- |
| Create / get a collection | `client.get_or_create_collection()` |
| Add documents | `collection.add()` |
| Similarity search | `collection.query()` |
| Metadata filtering | `where={...}`, `$or` / `$and`, `$contains` |
| Retrieve without search | `collection.get()` |
| Update | `collection.update()` |
| Delete documents | `collection.delete()` |
| Manage collections | `modify()`, `list_collections()`, `delete_collection()` |
| Reset everything | `client.reset()` |

Next steps: try embedding with a custom embedding function, experiment with
different `n_results`, or load a larger corpus and compare distances.